# Augmentation Ablation Study (SSL only)

This notebook trains SimCLR with different augmentation strategies and compares how well each strategy produces useful representations.

**Before running:** `Runtime → Change runtime type → T4 GPU`

### Why augmentation matters in SimCLR

SimCLR works by treating two different augmented views of the **same image** as a positive pair.  
The backbone must learn to produce similar embeddings for both views despite the augmentation.  
The augmentation directly defines **what invariances** the model is forced to learn:

| Strategy | What the model learns to ignore |
|---|---|
| `crop`   | Position, scale |
| `color`  | Brightness, contrast, saturation, hue |
| `rotate` | Orientation |
| `blur`   | Fine texture / sharpness |
| `noise`  | Pixel-level perturbations |
| `cutout` | Random occlusion |
| `sobel`  | Colour (only edge structure is preserved) |
| `full`   | All of the above simultaneously |
| `best`   | Crop + colour (SimCLR paper recommendation) |

### What you will get
- Overlaid **kNN accuracy curves** for all augmentation strategies
- Summary table of **final linear probing accuracy** per strategy
- A bar chart for easy visual comparison

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader
import torchvision.models as models
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}  ({vram:.1f} GB VRAM)")
else:
    DEVICE = torch.device('cpu')
    print("WARNING: No GPU. Go to Runtime → Change runtime type → GPU")

print(f"PyTorch: {torch.__version__}")

In [ ]:
USE_DRIVE = False   # Set True to save results to Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/AI_HW2/ablation_augmentation'
else:
    BASE_DIR = '/content/ablation_augmentation'

os.makedirs(f'{BASE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{BASE_DIR}/results',     exist_ok=True)
print(f"Working directory: {BASE_DIR}")

In [ ]:
import sys
from datetime import datetime

class _Tee:
    """Mirrors all print() output to both the notebook and a persistent log file."""
    def __init__(self, filepath):
        self._file   = open(filepath, 'a')
        self._stdout = sys.stdout
    def write(self, msg):
        self._stdout.write(msg)
        self._file.write(msg)
    def flush(self):
        self._stdout.flush()
        self._file.flush()
    def __enter__(self):
        sys.stdout = self
        return self
    def __exit__(self, *args):
        sys.stdout = self._stdout
        self._file.close()

LOG_FILE = f'{BASE_DIR}/results/training_log.txt'

with open(LOG_FILE, 'w') as f:
    f.write(f"Augmentation Ablation Log\n")
    f.write(f"Started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 60 + "\n\n")

print(f"Log file ready: {LOG_FILE}")

## 2. Config

**Edit `AUGMENTATIONS`** to choose which strategies to compare.  
Running all 9 on a T4 GPU with 100 epochs each takes roughly 2–3 hours.  
For a quick test, use a subset like `['crop', 'color', 'rotate', 'best']`.

In [ ]:
# -----------------------------------------------------------------------
# Augmentation strategies to compare — this is the variable being ablated
# -----------------------------------------------------------------------
AUGMENTATIONS = ['crop', 'cutout', 'color', 'sobel', 'noise', 'blur', 'rotate', 'full', 'best']

# -----------------------------------------------------------------------
# Fixed settings (same for all runs so the comparison is fair)
# -----------------------------------------------------------------------
DATA_DIR    = '/content/data'
NUM_CLASSES = 10
NUM_WORKERS = 2

SSL_EPOCHS       = 100     # Increase to 200 for better results (but 2× the time)
TEMPERATURE      = 0.5
SSL_BATCH_SIZE   = 256
SSL_LR           = 3e-4
SSL_WEIGHT_DECAY = 1e-6

PROJECTOR_HIDDEN_DIM = 512
PROJECTOR_OUT_DIM    = 128

KNN_K        = 20
KNN_INTERVAL = 5
KNN_TEMP     = 0.1

LP_EPOCHS       = 100
LP_LR           = 1e-3
LP_WEIGHT_DECAY = 1e-6

EVAL_BATCH_SIZE = 512

print(f"Will train {len(AUGMENTATIONS)} models: {AUGMENTATIONS}")
print(f"Each run: {SSL_EPOCHS} epochs, batch {SSL_BATCH_SIZE}, temperature {TEMPERATURE}")

## 3. Model, Augmentation, Loss & Evaluation Definitions

Self-contained — runs independently from the main notebook.

In [ ]:
# Custom tensor-level transforms used by some strategies

class _GaussianNoise:
    """Add isotropic Gaussian noise (applied after ToTensor)."""
    def __init__(self, std=0.05):
        self.std = std
    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std


class _SobelTransform:
    """
    Replace pixel content with Sobel edge magnitudes.
    Output: edge map in [0, 1], shape [3, H, W] (grayscale repeated).
    Dataset normalisation is skipped — Sobel replaces the image entirely.
    """
    def __call__(self, tensor):
        gray  = tensor.mean(dim=0, keepdim=True).unsqueeze(0)   # [1, 1, H, W]
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]],
                           dtype=torch.float32).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]],
                           dtype=torch.float32).view(1,1,3,3)
        ex = F.conv2d(gray, kx, padding=1)
        ey = F.conv2d(gray, ky, padding=1)
        edges = (ex**2 + ey**2).sqrt().squeeze(0)    # [1, H, W]
        edges = edges / (edges.max() + 1e-8)
        return edges.repeat(3, 1, 1)                 # [3, H, W]


print("Custom transforms defined.")

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)
IMAGE_SIZE   = 32

def _build_augmentation(name):
    """Return a Compose pipeline for the requested augmentation strategy."""
    normalize = transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)

    if name == 'crop':
        # Spatial variation only — no colour change.
        return transforms.Compose([
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            normalize,
        ])

    elif name == 'cutout':
        # Random rectangular occlusion via RandomErasing.
        return transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            normalize,
            transforms.RandomErasing(p=1.0, scale=(0.1, 0.33),
                                     ratio=(0.3, 3.3), value=0),
        ])

    elif name == 'color':
        # Colour variation only — no crop.
        return transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([
                transforms.ColorJitter(brightness=0.4, contrast=0.4,
                                       saturation=0.4, hue=0.1)
            ], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            normalize,
        ])

    elif name == 'sobel':
        # Edge map only — dataset normalization is skipped (Sobel output is in [0,1]).
        return transforms.Compose([
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            _SobelTransform(),
        ])

    elif name == 'noise':
        # Additive Gaussian noise only.
        return transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            normalize,
            _GaussianNoise(std=0.05),
        ])

    elif name == 'blur':
        # Gaussian smoothing — tests whether coarse structure alone is sufficient.
        return transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
            transforms.ToTensor(),
            normalize,
        ])

    elif name == 'rotate':
        # Random rotation ± 30°.
        return transforms.Compose([
            transforms.RandomRotation(degrees=30),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            normalize,
        ])

    elif name == 'full':
        # Everything: crop + rotation + flip + colour + grayscale + blur + noise + cutout.
        return transforms.Compose([
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.2, 1.0)),
            transforms.RandomRotation(degrees=15),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([
                transforms.ColorJitter(brightness=0.4, contrast=0.4,
                                       saturation=0.4, hue=0.1)
            ], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.RandomApply([
                transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
            ], p=0.5),
            transforms.ToTensor(),
            normalize,
            transforms.RandomErasing(p=0.5, scale=(0.02, 0.15)),
            _GaussianNoise(std=0.02),
        ])

    elif name == 'best':
        # SimCLR paper optimal: crop + flip + colour + grayscale.
        return transforms.Compose([
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([
                transforms.ColorJitter(brightness=0.4, contrast=0.4,
                                       saturation=0.4, hue=0.1)
            ], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            normalize,
        ])

    else:
        raise ValueError(f"Unknown augmentation: '{name}'")


class SimCLRTransform:
    """Applies two independent views using the chosen augmentation strategy."""
    def __init__(self, augmentation='best'):
        self.augment = _build_augmentation(augmentation)
    def __call__(self, image):
        return self.augment(image), self.augment(image)


EVAL_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
])


def get_ssl_loader(augmentation):
    dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=True,
                                transform=SimCLRTransform(augmentation))
    return DataLoader(dataset, batch_size=SSL_BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)


def get_eval_loaders():
    train_set = datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=EVAL_TRANSFORM)
    test_set  = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=EVAL_TRANSFORM)
    return (DataLoader(train_set, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True),
            DataLoader(test_set,  batch_size=EVAL_BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True))


print("Augmentation factory and data loaders defined.")

In [ ]:
class ModifiedResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=None)
        resnet.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()
        self.encoder   = nn.Sequential(*list(resnet.children())[:-1])
        self.output_dim = 512
    def forward(self, x):
        return torch.flatten(self.encoder(x), start_dim=1)


class ProjectorHead(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=512, out_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x): return self.mlp(x)


class SimCLRModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone  = ModifiedResNet18()
        self.projector = ProjectorHead(512, PROJECTOR_HIDDEN_DIM, PROJECTOR_OUT_DIM)
    def forward(self, x):
        h = self.backbone(x)
        return h, self.projector(h)
    def encode(self, x):
        return self.backbone(x)


class NTXentLoss(nn.Module):
    def __init__(self, temperature):
        super().__init__()
        self.temperature = temperature
    def forward(self, z_i, z_j):
        N   = z_i.shape[0]
        z   = torch.cat([F.normalize(z_i, dim=1), F.normalize(z_j, dim=1)], dim=0)
        sim = torch.mm(z, z.T) / self.temperature
        labels = torch.cat([torch.arange(N, device=z_i.device) + N,
                             torch.arange(N, device=z_i.device)])
        sim = sim.masked_fill(torch.eye(2*N, dtype=torch.bool, device=z_i.device), float('-inf'))
        return F.cross_entropy(sim, labels)


print("Model and loss defined.")

In [ ]:
@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    feats, labs = [], []
    for images, labels in loader:
        feats.append(model.encode(images.to(DEVICE)).cpu())
        labs.append(labels)
    return torch.cat(feats), torch.cat(labs)


@torch.no_grad()
def knn_monitor(model, train_loader, test_loader):
    train_feats, train_labels = extract_features(model, train_loader)
    test_feats,  test_labels  = extract_features(model, test_loader)
    train_feats  = F.normalize(train_feats,  dim=1).to(DEVICE)
    test_feats   = F.normalize(test_feats,   dim=1).to(DEVICE)
    train_labels = train_labels.to(DEVICE)
    correct = total = 0
    for start in range(0, len(test_feats), 512):
        end          = min(start + 512, len(test_feats))
        batch_feats  = test_feats[start:end]
        batch_labels = test_labels[start:end].to(DEVICE)
        sim          = torch.mm(batch_feats, train_feats.T) / KNN_TEMP
        topk_v, topk_idx = sim.topk(KNN_K, dim=1)
        weights      = F.softmax(topk_v, dim=1)
        neighbor_labs = train_labels[topk_idx]
        votes        = torch.zeros(end - start, NUM_CLASSES, device=DEVICE)
        votes.scatter_add_(1, neighbor_labs, weights)
        correct += (votes.argmax(1) == batch_labels).sum().item()
        total   += len(batch_labels)
    return correct / total


def linear_probing(model, train_loader, test_loader):
    for p in model.backbone.parameters(): p.requires_grad = False
    model.eval()
    linear    = nn.Linear(512, NUM_CLASSES).to(DEVICE)
    optimizer = Adam(linear.parameters(), lr=LP_LR, weight_decay=LP_WEIGHT_DECAY)
    loss_fn   = nn.CrossEntropyLoss()
    for _ in range(LP_EPOCHS):
        linear.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.no_grad(): features = model.encode(images)
            loss = loss_fn(linear(features), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    linear.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            correct += (linear(model.encode(images)).argmax(1) == labels).sum().item()
            total   += len(labels)
    for p in model.backbone.parameters(): p.requires_grad = True
    return correct / total


print("Evaluation utilities defined.")

## 4. Run All Augmentation Experiments

Each augmentation strategy gets its own SSL training run from scratch.  
The eval loaders (used for kNN and linear probing) are shared across all runs.

In [ ]:
eval_train_loader, eval_test_loader = get_eval_loaders()

all_histories = {}

for aug_name in AUGMENTATIONS:
    ssl_loader = get_ssl_loader(aug_name)
    model      = SimCLRModel().to(DEVICE)
    criterion  = NTXentLoss(temperature=TEMPERATURE)
    optimizer  = Adam(model.parameters(), lr=SSL_LR, weight_decay=SSL_WEIGHT_DECAY)
    history    = {'loss': [], 'knn_acc': []}

    with _Tee(LOG_FILE):
        print(f"\n{'#'*60}")
        print(f"  Augmentation: {aug_name}")
        print(f"{'#'*60}")

        for epoch in range(1, SSL_EPOCHS + 1):
            model.train()
            total_loss = 0.0
            for (view1, view2), _ in ssl_loader:
                view1, view2 = view1.to(DEVICE), view2.to(DEVICE)
                _, z_i = model(view1)
                _, z_j = model(view2)
                loss   = criterion(z_i, z_j)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                total_loss += loss.item()

            avg_loss = total_loss / len(ssl_loader)
            history['loss'].append(avg_loss)

            run_knn = (epoch % KNN_INTERVAL == 0)
            if run_knn:
                acc = knn_monitor(model, eval_train_loader, eval_test_loader)
                history['knn_acc'].append((epoch, acc))

            if epoch % 10 == 0 or epoch == 1:
                knn_str = f"  kNN: {acc*100:.2f}%" if run_knn and epoch % 10 == 0 else ""
                print(f"  Epoch {epoch:3d}/{SSL_EPOCHS}  loss={avg_loss:.4f}{knn_str}")

        ckpt_path = f'{BASE_DIR}/checkpoints/simclr_{aug_name}.pt'
        torch.save(model.state_dict(), ckpt_path)
        print(f"  Saved: {ckpt_path}")

    all_histories[aug_name] = history

print(f"\nAll {len(AUGMENTATIONS)} runs complete.")

## 5. Comparison Plots

In [ ]:
# 9 visually distinct colours
COLORS = [
    '#e63946', '#2a9d8f', '#e9c46a', '#457b9d', '#f4a261',
    '#6a4c93', '#80b918', '#fb8500', '#8ecae6'
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SimCLR Augmentation Ablation Study', fontsize=14, fontweight='bold')

for i, aug_name in enumerate(AUGMENTATIONS):
    history = all_histories[aug_name]
    color   = COLORS[i % len(COLORS)]

    axes[0].plot(range(1, len(history['loss']) + 1), history['loss'],
                 color=color, label=aug_name, linewidth=1.8)

    if history['knn_acc']:
        epochs, accs = zip(*history['knn_acc'])
        axes[1].plot(epochs, [a * 100 for a in accs],
                     color=color, label=aug_name, linewidth=1.8,
                     marker='o', markersize=3)

axes[0].set_xlabel('Epoch');  axes[0].set_ylabel('NT-Xent Loss')
axes[0].set_title('Training Loss')
axes[0].legend(fontsize=8, loc='upper right'); axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Epoch');  axes[1].set_ylabel('kNN Accuracy (%)')
axes[1].set_title(f'kNN Monitor (k={KNN_K})')
axes[1].legend(fontsize=8, loc='lower right'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = f'{BASE_DIR}/results/augmentation_ablation_curves.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved: {plot_path}")

## 6. Linear Probing Comparison

In [ ]:
lp_results = {}

for aug_name in AUGMENTATIONS:
    model = SimCLRModel().to(DEVICE)
    model.load_state_dict(torch.load(
        f'{BASE_DIR}/checkpoints/simclr_{aug_name}.pt',
        map_location=DEVICE, weights_only=True))
    with _Tee(LOG_FILE):
        print(f"\n--- Linear Probing: {aug_name} ---")
        acc = linear_probing(model, eval_train_loader, eval_test_loader)
        print(f"    Accuracy: {acc*100:.2f}%")
    lp_results[aug_name] = acc

with _Tee(LOG_FILE):
    print(f"\n{'='*55}")
    print(f"  Augmentation Ablation — Final Results")
    print(f"{'='*55}")
    print(f"  {'Augmentation':>12}  {'Linear Probe Acc':>18}")
    print(f"  {'-'*12}  {'-'*18}")
    for aug_name in AUGMENTATIONS:
        marker = '  ← paper best' if aug_name == 'best' else ''
        print(f"  {aug_name:>12}  {lp_results[aug_name]*100:>17.2f}%{marker}")
    print(f"{'='*55}")

with open(f'{BASE_DIR}/results/augmentation_summary.txt', 'w') as f:
    f.write(f"Augmentation Ablation Study\n")
    f.write(f"SSL epochs: {SSL_EPOCHS}, temperature: {TEMPERATURE}, batch: {SSL_BATCH_SIZE}\n\n")
    f.write(f"{'Augmentation':>12}  {'Linear Probe Acc':>18}\n")
    for aug_name in AUGMENTATIONS:
        f.write(f"{aug_name:>12}  {lp_results[aug_name]*100:>17.2f}%\n")
print(f"Summary saved to {BASE_DIR}/results/augmentation_summary.txt")

In [ ]:
# Bar chart for easy visual comparison
names = list(lp_results.keys())
accs  = [lp_results[n] * 100 for n in names]
bar_colors = [COLORS[i % len(COLORS)] for i in range(len(names))]

# Highlight 'best' with a border
edge_colors = ['black' if n == 'best' else 'none' for n in names]
line_widths = [2.0  if n == 'best' else 0.0  for n in names]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(names, accs, color=bar_colors,
              edgecolor=edge_colors, linewidth=line_widths)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Linear Probe Accuracy (%)')
ax.set_title('SimCLR Augmentation Ablation — Linear Probe Accuracy', fontweight='bold')
ax.set_ylim(0, max(accs) * 1.12)
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
bar_path = f'{BASE_DIR}/results/augmentation_ablation_bar.png'
plt.savefig(bar_path, dpi=150)
plt.show()
print(f"Saved: {bar_path}")

## Notes for your report

**What to look for and discuss:**

- **`best` (crop + colour):** Should achieve the highest accuracy. The SimCLR paper (Chen et al. 2020, Figure 5) shows that combining spatial and colour invariances is far more powerful than either alone. This is your key finding to reference.

- **`crop` vs `color`:** Both are useful on their own, but neither alone reaches `best`. Crop forces spatial invariance; colour forces colour invariance. Together they eliminate two orthogonal cues, forcing the model to rely on semantic content.

- **`rotate`:** Expected to underperform `crop`. Rotation produces very similar views (both show the same object at similar scale) so the contrastive task is too easy — the model doesn't need to learn much semantic structure to solve it.

- **`sobel`:** An interesting extreme — the model only sees edge structure, not colour or texture. Two random crops of the same edge map are still quite similar, limiting diversity. Likely to underperform `best` significantly.

- **`full`:** May not beat `best` despite including more augmentations. When too many invariances are enforced simultaneously, the views can become so different that the model cannot find useful shared features (the signal-to-noise ratio drops). This is a known trade-off discussed in SimCLR follow-up work.

- **`cutout`, `noise`, `blur`:** Weaker invariances in isolation. Useful as components but insufficient alone, because two views with the same cutout location or blur level still look very similar.

**Cite:** Chen et al., "A Simple Framework for Contrastive Learning of Visual Representations," ICML 2020.